In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import lower
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer,
    HashingTF,
    IDF,
    ChiSqSelector,
    Normalizer,
    StringIndexer,
    IndexToString
)
from pyspark.ml.classification import LinearSVC, OneVsRest # What is this for? We are only supposed to use SVM
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, TrainValidationSplit

In [ ]:
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()

In [ ]:
# Set strings and flags
dev_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
full_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviewscombined.json"
stopwords_path = "../data/stopwords.txt"

pattern = "[ \t0-9\\(\\)\\[\\]\\{\\}\\.\\!\\?,;:\\+=\\-_\"'`~#@&%*%€$§\\\\/|<>^]+"

DEVEL = True

In [ ]:
# Read data according to flag
dataframe = spark.read.json(dev_set_path) if DEVEL else spark.read.json(full_set_path)

# Load stopwords into a set
with open(stopwords_path, "r") as f:
    stopwords = set(line.strip() for line in f)
stopwords.add("")

In [ ]:
def project_relevant_cols(data:DataFrame):
    return data.select(lower(data.reviewText).alias("reviewText"), "category").na.drop() # Are there NAs? I don't think so but whatever

In [ ]:
dataframe = project_relevant_cols(dataframe)

In [ ]:
dataframe.show()

In [ ]:
# StringIndexer, transforms the category names to labels (0 to n_categories)
str_idx = StringIndexer(inputCol="category", outputCol="label")

In [ ]:
# Regex tokenizer, to split review texts into words
rgx_tknizer = RegexTokenizer(inputCol="reviewText", outputCol="Words", pattern=pattern, gaps=True, minTokenLength =2)

In [ ]:
# Stopword remover, loaded with stopwords.txt
stop_words_remover = StopWordsRemover(inputCol="Words", outputCol="filtered_words", stopWords=list(stopwords))

In [ ]:
# Term document frequency counter
vectorizer = CountVectorizer(inputCol="filtered_words", outputCol="vectorized_Features")

In [ ]:
# Inverse doc freq
idf = IDF(inputCol="vectorized_Features", outputCol="features")

In [ ]:
# Scores each term according to Chi test. Finds top relevant features
chi_selector = ChiSqSelector(numTopFeatures=2000, featuresCol="features", outputCol="selectedFeatures", labelCol="label")

In [ ]:
# Build the pipeline as concatenation of stages. Output col of previous serves as input col for next
pipeline = Pipeline(stages=[str_idx, rgx_tknizer, stop_words_remover, vectorizer, idf, chi_selector])

In [ ]:
# Fit pipeline to our data
fitted_pipeline = pipeline.fit(dataframe) # I don't think you need to create another object...

In [ ]:
# Transform data
trasnsformed_data = fitted_pipeline.transform(dataframe) 
# Is this necessary? We are only interested in the terms with highest chi, not in what the final output looks like...

In [ ]:
dataframe.unpersist()

In [ ]:
chi_selector_model = fitted_pipeline.stages[5]

# Get indices for 2000 top features
selected_indices = chi_selector_model.selectedFeatures

# All vocabulary
vocabulary = fitted_pipeline.stages[3].vocabulary

# Top terms
selected_words = [vocabulary[i] for i in selected_indices]

sorted_words = sorted(selected_words)

words_line = "\n".join(sorted_words)

with open("output_ds.txt", "w") as file:
    file.write(words_line)

In [ ]:
spark.stop()